# Tạo mô tả cho ảnh test trên Kaggle

Notebook tạo một mô tả JSON cho mỗi `image_id`. Prompt chỉ chứa ảnh và instruction mô tả cố định; không dùng câu hỏi, lựa chọn, đáp án hay ngữ cảnh luật.

## 1. Cấu hình

Thêm dataset chứa thư mục ảnh vào notebook, sau đó điền `IMAGES_DIR` bên dưới.

In [ ]:
from dataclasses import dataclass
import json
from pathlib import Path

IMAGES_DIR = ""
RESUME_PATH = ""
OUTPUT_PATH = "/kaggle/working/img_to_txt_test_descriptions.json"
ERRORS_PATH = "/kaggle/working/img_to_txt_test_description_errors.json"
MODEL_ID = "Qwen/Qwen3.5-4B"
LIMIT = None
RETRY_ERRORS = False

CATEGORIES = {"SIGN_ILLUSTRATION", "TECHNICAL_DIAGRAM", "MARKING_AND_SIGNAL"}
DESCRIPTION_FIELDS = {"image_category", "summary", "items"}
ITEM_FIELDS = {"identifier", "visual_or_geometric_features", "meaning_or_specification"}


In [ ]:
@dataclass(frozen=True)
class TestImageJob:
    image_id: str
    image_path: Path


def discover_image_jobs(images_dir: Path) -> list[TestImageJob]:
    image_paths = sorted(
        (path for path in images_dir.iterdir() if path.is_file() and path.suffix.lower() == ".jpg"),
        key=lambda path: path.name.casefold(),
    )
    if not image_paths:
        raise ValueError(f"No JPG images found in {images_dir}")
    jobs = []
    seen_image_ids = set()
    for image_path in image_paths:
        image_id = image_path.stem
        if image_id in seen_image_ids:
            raise ValueError(f"Duplicate image_id from filenames: {image_id}")
        seen_image_ids.add(image_id)
        jobs.append(TestImageJob(image_id=image_id, image_path=image_path))
    return jobs


def build_image_only_prompt() -> str:
    return '''/no_think
Bạn là chuyên gia mô tả hình ảnh giao thông Việt Nam. Hãy mô tả khách quan chỉ từ những gì quan sát được trong ảnh. Chú ý chữ, số, ký hiệu, màu sắc, hình dạng và bố cục. Không bịa chi tiết không nhìn rõ và không viện dẫn điều luật cụ thể.

Chọn đúng một image_category: SIGN_ILLUSTRATION, TECHNICAL_DIAGRAM hoặc MARKING_AND_SIGNAL. Chỉ trả về một object JSON hợp lệ, không Markdown, với đúng schema sau:
{
  "image_category": "SIGN_ILLUSTRATION",
  "summary": "Mô tả ngắn gọn toàn ảnh",
  "items": [
    {
      "identifier": "Tên, mã hoặc ký hiệu nhìn thấy",
      "visual_or_geometric_features": "Màu sắc, hình dạng và đặc điểm trực quan",
      "meaning_or_specification": "Ý nghĩa có khả năng phù hợp chỉ từ ảnh"
    }
  ]
}
Summary tối đa 2 câu; items tối đa 5 phần tử; mỗi field trong items tối đa 25 từ. Kết thúc ngay sau dấu } cuối cùng.'''


def parse_description(raw: str) -> dict[str, object]:
    object_start = raw.find("{")
    if object_start < 0:
        raise ValueError("Model response does not contain a JSON object")
    try:
        value, _ = json.JSONDecoder().raw_decode(raw[object_start:])
    except json.JSONDecodeError as error:
        raise ValueError(f"Invalid JSON response: {error}") from error
    if not isinstance(value, dict) or set(value) != DESCRIPTION_FIELDS:
        raise ValueError(f"description fields must be exactly {sorted(DESCRIPTION_FIELDS)}")
    if value["image_category"] not in CATEGORIES:
        raise ValueError("image_category is not supported")
    if not isinstance(value["summary"], str) or not value["summary"].strip():
        raise ValueError("summary must be a non-empty string")
    if not isinstance(value["items"], list):
        raise ValueError("items must be a list")
    for item in value["items"]:
        if not isinstance(item, dict) or set(item) != ITEM_FIELDS:
            raise ValueError(f"item fields must be exactly {sorted(ITEM_FIELDS)}")
        if not all(isinstance(item[field], str) for field in ITEM_FIELDS):
            raise ValueError("all item fields must be strings")
    return value


In [ ]:
def new_cache(model_id: str) -> dict[str, object]:
    return {"schema_version": "1.0", "model_id": model_id, "descriptions": {}}


def save_json(path: Path, payload: dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary_path.replace(path)


def save_cache(path: Path, payload: dict[str, object]) -> None:
    save_json(path, payload)


def load_cache(path: Path, model_id: str) -> dict[str, object]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict) or payload.get("schema_version") != "1.0":
        raise ValueError("cache schema_version must be '1.0'")
    if payload.get("model_id") != model_id:
        raise ValueError(f"cache model_id is {payload.get('model_id')!r}; expected {model_id!r}")
    if set(payload) != {"schema_version", "model_id", "descriptions"}:
        raise ValueError("cache must contain exactly schema_version, model_id, and descriptions")
    if not isinstance(payload.get("descriptions"), dict):
        raise ValueError("cache descriptions must be an object")
    return payload


def prepare_cache(output_path: Path, resume_path: Path | None, model_id: str) -> dict[str, object]:
    if output_path.exists():
        return load_cache(output_path, model_id)
    if resume_path is not None:
        if not resume_path.is_file():
            raise FileNotFoundError(f"Resume checkpoint not found: {resume_path}")
        payload = load_cache(resume_path, model_id)
        save_cache(output_path, payload)
        return payload
    return new_cache(model_id)


def new_errors(model_id: str) -> dict[str, object]:
    return {"schema_version": "1.0", "model_id": model_id, "errors": {}}


def load_errors(path: Path, model_id: str) -> dict[str, object]:
    if not path.exists():
        return new_errors(model_id)
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict) or payload.get("schema_version") != "1.0":
        raise ValueError("errors schema_version must be '1.0'")
    if payload.get("model_id") != model_id:
        raise ValueError("errors model_id does not match configured model")
    if not isinstance(payload.get("errors"), dict):
        raise ValueError("errors must be an object")
    return payload


def run_jobs(
    jobs: list[TestImageJob],
    output_path: Path,
    errors_path: Path,
    cache: dict[str, object],
    describe,
    limit: int | None,
    retry_errors: bool,
    on_error=None,
) -> dict[str, int]:
    errors_payload = load_errors(errors_path, str(cache["model_id"]))
    completed = set(cache["descriptions"])
    if not retry_errors:
        completed.update(errors_payload["errors"])
    pending = [job for job in jobs if job.image_id not in completed]
    if limit is not None:
        pending = pending[:limit]
    save_cache(output_path, cache)
    save_json(errors_path, errors_payload)
    stats = {"attempted": 0, "succeeded": 0, "failed": 0}
    for current, job in enumerate(pending, start=1):
        print(f"[{current}/{len(pending)}] {job.image_id}", flush=True)
        stats["attempted"] += 1
        try:
            cache["descriptions"][job.image_id] = describe(job)
            errors_payload["errors"].pop(job.image_id, None)
            stats["succeeded"] += 1
        except Exception as exception:
            error_message = f"{type(exception).__name__}: {exception}"
            errors_payload["errors"][job.image_id] = error_message
            stats["failed"] += 1
            print(f"  Error: {error_message}", flush=True)
            if on_error is not None:
                on_error(job, exception)
        save_cache(output_path, cache)
        save_json(errors_path, errors_payload)
    return stats


In [ ]:
def generate_description(
    job: TestImageJob, processor: object, model: object
) -> dict[str, object]:
    prompt = build_image_only_prompt()
    for attempt in range(2):
        current_prompt = prompt
        if attempt == 1:
            current_prompt += (
                "\nCâu trả lời trước không phải JSON đúng schema. "
                "Hãy trả lại đúng một object JSON hợp lệ."
            )
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "path": str(job.image_path.resolve())},
                {"type": "text", "text": current_prompt},
            ],
        }]
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=False,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)
        generated_ids = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=384 if attempt == 0 else 768,
        )
        input_length = len(inputs["input_ids"][0])
        generated_only = [row[input_length:] for row in generated_ids]
        raw_response = processor.batch_decode(
            generated_only,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
        try:
            return parse_description(raw_response)
        except ValueError:
            if attempt == 1:
                raise
    raise RuntimeError("unreachable")


## 2. Kiểm tra input

Cell này quét trực tiếp toàn bộ file JPG trong `IMAGES_DIR` trước khi nạp model.

In [ ]:
if not IMAGES_DIR.strip():
    raise ValueError("Hãy điền IMAGES_DIR trong ô cấu hình.")

images_dir = Path(IMAGES_DIR)
resume_path = Path(RESUME_PATH) if RESUME_PATH.strip() else None
output_path = Path(OUTPUT_PATH)
errors_path = Path(ERRORS_PATH)

if not images_dir.is_dir():
    raise NotADirectoryError(f"Không tìm thấy thư mục ảnh: {images_dir}")

jobs = discover_image_jobs(images_dir)
print(f"Đã tìm thấy {len(jobs)} ảnh JPG.")


## 3. Nạp Qwen

Bật GPU Accelerator trong Kaggle. Model được nạp 4-bit NF4; GPU hỗ trợ tốt BF16 sẽ dùng `bfloat16`, còn lại dùng `float16`.

In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3_5ForConditionalGeneration

if not torch.cuda.is_available():
    raise RuntimeError("Notebook cần Kaggle GPU. Hãy bật Accelerator trong Settings.")

major_capability, _ = torch.cuda.get_device_capability(0)
compute_dtype = torch.bfloat16 if major_capability >= 8 else torch.float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=compute_dtype,
)
model.eval()
print(f"Đã nạp {MODEL_ID} trên {torch.cuda.get_device_name(0)} với {compute_dtype}.")


## 4. Sinh mô tả và checkpoint

Cache thành công được ghi đúng schema mà `evaluate.py --description-cache` đọc. Lỗi được ghi riêng để cache chính luôn tương thích.

In [ ]:
def release_cuda_after_error(job: TestImageJob, exception: Exception) -> None:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


cache = prepare_cache(output_path, resume_path, MODEL_ID)
print(f"Cache hiện có {len(cache['descriptions'])} mô tả.")
stats = run_jobs(
    jobs,
    output_path,
    errors_path,
    cache,
    lambda job: generate_description(job, processor, model),
    LIMIT,
    RETRY_ERRORS,
    on_error=release_cuda_after_error,
)
print(f"Kết quả lần chạy: {stats}")
print(f"Tổng mô tả thành công: {len(cache['descriptions'])}")
print(f"File output: {output_path}")


## 5. Kiểm tra output

Sau khi lưu phiên notebook, tải file JSON từ mục Kaggle Output.

In [ ]:
result = load_cache(output_path, MODEL_ID)
print(json.dumps({
    "schema_version": result["schema_version"],
    "model_id": result["model_id"],
    "description_count": len(result["descriptions"]),
}, ensure_ascii=False, indent=2))
if result["descriptions"]:
    last_image_id = next(reversed(result["descriptions"]))
    print(last_image_id)
    print(json.dumps(result["descriptions"][last_image_id], ensure_ascii=False, indent=2))
